# 🎛️ Hyperparameter Optimization: Hard Mining Strategy
## Bayesian Search for U-Net Architecture & Loss Balancing

**Author:** Ibon Garcia Gomez  
**Objective:** Fine-tune the Deep U-Net architecture to handle the "Hard Mining" dataset.  
**Methodology:** Tree-structured Parzen Estimator (TPE) via Optuna framework.

---

### 🧪 Experiment Hypothesis
The introduction of "Hard Samples" (low SNR streams and dense cirrus masking) changes the optimal decision boundary of the network. We hypothesize that:
1.  **Dice Weight:** The loss function may need to prioritize shape (Dice) over pixel accuracy to recover faint streams.
2.  **Regularization:** Higher Dropout rates might be needed to prevent overfitting on noise.
3.  **Depth:** We need to determine if a deeper network (Depth=5) is required to disentangle spectral overlapping.

### ⚙️ Strategy
* **Subset Training:** We use a representative subset of 4,000 images (Normal + Hard) to accelerate the search.
* **Pruning:** We use `HyperbandPruner` to terminate unpromising trials early (e.g., if a model doesn't converge by epoch 3).

In [1]:
!pip install optuna
!pip install optuna-integration[tfkeras]
!pip install astropy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.9/319.9 kB 15.8 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-cloud-translate 3.12.1 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<5.0.0dev,>=3.19.5, but you have protobuf 5.29.5 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
pydrive2 1.21.3 requires cryptograph

## ⚡ Environment Setup & Acceleration
Mixed Precision (float16) is enabled to reduce memory usage and accelerate training by ~2× on Tensor Core GPUs. The configuration defines key constraints for the search: a **4,000-sample subset**, **30 trials**, and a **7-hour timeout**.

In [2]:
import os
import gc
import optuna
import numpy as np
import tensorflow as tf
from astropy.io import fits
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import mixed_precision
from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, Activation, MaxPooling2D, Dropout, concatenate, UpSampling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tqdm.notebook import tqdm

# ==========================================
# ⚡ 1. ACCELERATION & CONFIG
# ==========================================
# Enable Mixed Precision for ~2x speedup during optimization
try:
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    print("🚀 Mixed Precision Enabled.")
except:
    pass

class Config:
    IMG_SIZE = (128, 128)
    N_CHANNELS = 4
    N_CLASSES = 3
    
    # Ensure this path points to the dataset containing 'images' and 'masks' folders
    DATASET_DIR = '/kaggle/input/synthetic-dataset/synthetic_dataset' 
    
    # Optimization Constraints
    SAMPLE_LIMIT = 4000    # Use a random 4k subset for rapid optimization
    N_TRIALS = 30          # Total number of experiments
    TIMEOUT_HOURS = 7      # Maximum runtime
    EPOCHS_PER_TRIAL = 15  # Short runs to test convergence speed

2025-12-11 16:07:00.919422: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765469221.095728      19 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765469221.152890      19 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


🚀 Mixed Precision Enabled.


In [3]:
# ==========================================
# 📥 2. DATA LOADER (SINGLE SOURCE SUBSET)
# ==========================================
def load_optimization_subset(root_path, limit=Config.SAMPLE_LIMIT):
    """
    Loads a random subset from the unified dataset for hyperparameter tuning.
    Random sampling ensures representation of both Normal and Hard cases.
    """
    X_list, Y_list = [], []
    
    img_dir = os.path.join(root_path, 'images')
    mask_dir = os.path.join(root_path, 'masks')
    
    if not os.path.exists(img_dir):
        raise FileNotFoundError(f"❌ Images folder not found at: {root_path}")
        
    print(f"🔍 Scanning {root_path}...")
    
    # List all files
    all_files = sorted([f for f in os.listdir(img_dir) if f.endswith('.fits')])
    total_files = len(all_files)
    
    if total_files == 0:
        raise ValueError("❌ Dataset is empty.")
        
    print(f"📦 Found {total_files} total images. Selecting a random subset of {limit}...")
    
    # Shuffle and select subset
    np.random.seed(42) # Ensure reproducibility
    np.random.shuffle(all_files)
    selected_files = all_files[:limit]
    
    for f in tqdm(selected_files, desc="Loading Data"):
        try:
            # Load Image
            img_p = os.path.join(img_dir, f)
            with fits.open(img_p) as h:
                d = h[0].data.astype(np.float32)
                if d.shape[0]==4: d = np.moveaxis(d, 0, -1)
                # Robust Normalization
                p1, p99 = np.percentile(d, 1), np.percentile(d, 99)
                d = np.clip((d - p1)/(p99 - p1), 0, 1)
                X_list.append(d)
            
            # Load Mask (Standardize naming convention: img_ -> mask_)
            f_mask = f.replace('img_', 'mask_').replace('image_', 'mask_')
            mask_p = os.path.join(mask_dir, f_mask)
            
            # Fallback if names are identical
            if not os.path.exists(mask_p):
                mask_p = os.path.join(mask_dir, f)
            
            with fits.open(mask_p) as h:
                Y_list.append(to_categorical(h[0].data, 3))
        except Exception as e:
            continue
            
    return np.array(X_list), np.array(Y_list)

# Load Data & Split
X, Y = load_optimization_subset(Config.DATASET_DIR)
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

# Garbage Collection
del X, Y
gc.collect()

print(f"✅ Optimization Dataset Ready: {len(X_train)} Training | {len(X_val)} Validation")

🔍 Scanning /kaggle/input/synthetic-dataset/synthetic_dataset...
📦 Found 15000 total images. Selecting a random subset of 4000...


Loading Data:   0%|          | 0/4000 [00:00<?, ?it/s]

✅ Optimization Dataset Ready: 3200 Training | 800 Validation


## 🧠 Dynamic U-Net Builder & Objective Function
The `build_dynamic_unet` function creates a U-Net where **depth** (3–5), **start filters** (32/64), and **dropout** (0.2–0.5) are trial variables. The objective function also tunes:
- **Learning Rate** (log-uniform in `[1e-4, 1e-3]`)
- **Dice Weight** (`[0.5, 0.99]`) — balancing shape recovery vs. pixel accuracy

Each trial trains for 15 epochs. `HyperbandPruner` terminates unpromising trials early based on `val_dice_metric`.

In [4]:
# ==========================================
# 🧠 3. DYNAMIC U-NET BUILDER
# ==========================================
def build_dynamic_unet(trial):
    """
    Constructs a U-Net where Depth, Filters, and Dropout are flexible variables.
    """
    # Suggest Hyperparameters
    start_filters = trial.suggest_categorical('start_filters', [32, 64])
    dropout = trial.suggest_float('dropout', 0.2, 0.5)
    depth = trial.suggest_int('depth', 3, 5) # Test shallow vs deep networks
    
    inputs = Input((128, 128, 4))
    x = inputs
    skips = []
    filters = start_filters
    
    # --- Encoder ---
    for i in range(depth):
        x = Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
        x = BatchNormalization()(x); x = Activation('relu')(x)
        x = Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
        x = BatchNormalization()(x); x = Activation('relu')(x)
        
        skips.append(x)
        x = MaxPooling2D(2)(x)
        x = Dropout(dropout)(x)
        filters *= 2
        
    # --- Bridge ---
    x = Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x); x = Activation('relu')(x)
    
    # --- Decoder ---
    for i in reversed(range(depth)):
        filters //= 2
        x = UpSampling2D(2)(x)
        x = concatenate([x, skips[i]])
        x = Dropout(dropout)(x)
        
        x = Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
        x = BatchNormalization()(x); x = Activation('relu')(x)
        x = Conv2D(filters, 3, padding='same', kernel_initializer='he_normal')(x)
        x = BatchNormalization()(x); x = Activation('relu')(x)
        
    # Output (float32 for stability)
    outputs = Conv2D(Config.N_CLASSES, 1, activation='softmax', dtype='float32')(x)
    
    return Model(inputs, outputs)

In [5]:
# ==========================================
# 🎯 4. OBJECTIVE FUNCTION 
# ==========================================
def objective(trial):
    # 1. Clear clutter to prevent OOM (Memory leaks)
    tf.keras.backend.clear_session()
    gc.collect() # Force garbage collection
    
    # 2. Sample Hyperparameters
    lr = trial.suggest_float('learning_rate', 1e-4, 1e-3, log=True)
    
    # CRITICAL: Balancing Shape vs Pixel Accuracy
    dice_weight = trial.suggest_float('dice_weight', 0.5, 0.99) 
    
    # 3. Define Loss
    def dice_metric(y_true, y_pred, smooth=1e-6):
        y_true = tf.cast(y_true, tf.float32); y_pred = tf.cast(y_pred, tf.float32)
        inter = tf.reduce_sum(y_true * y_pred, axis=(1,2))
        union = tf.reduce_sum(y_true, axis=(1,2)) + tf.reduce_sum(y_pred, axis=(1,2))
        return tf.reduce_mean((2. * inter + smooth) / (union + smooth))
        
    def hybrid_loss_dynamic(y_true, y_pred):
        cce = CategoricalCrossentropy()(y_true, y_pred)
        dice = 1.0 - dice_metric(y_true, y_pred)
        return ((1.0 - dice_weight) * cce) + (dice_weight * dice)
    
    # 4. Build & Compile
    try:
        model = build_dynamic_unet(trial)
        model.compile(
            optimizer=Adam(learning_rate=lr), 
            loss=hybrid_loss_dynamic, 
            metrics=[dice_metric]
        )
        
        # 5. Train with Pruning & SAFETY CATCH
        # We use Batch Size 16 to be safe with large models (Depth 5 + 64 filters)
        history = model.fit(
            X_train, Y_train,
            validation_data=(X_val, Y_val),
            batch_size=16,  
            epochs=Config.EPOCHS_PER_TRIAL,
            verbose=1,
            callbacks=[
                optuna.integration.TFKerasPruningCallback(trial, 'val_dice_metric')
            ]
        )
        
        # Maximizing Validation Dice Score
        val_dice = max(history.history['val_dice_metric'])
        return val_dice

    except tf.errors.ResourceExhaustedError:
        # 🛡️ SAFETY NET: If GPU runs out of memory, don't crash the script.
        # Just prune this trial and continue with the next one.
        print(f"⚠️ Trial {trial.number} skipped: OOM (Model too large for GPU).")
        tf.keras.backend.clear_session()
        gc.collect()
        raise optuna.exceptions.TrialPruned()
        
    except Exception as e:
        # Handle other errors gracefully
        print(f"⚠️ Trial {trial.number} failed: {e}")
        tf.keras.backend.clear_session()
        raise e

## 🚀 Optimization Execution
The study runs for up to 30 trials (or 7 hours) using the **TPE (Tree-Structured Parzen Estimator)** sampler. Results are exported to CSV for analysis and the best hyperparameters are printed at completion.

In [6]:
# ==========================================
# 🚀 5. RUN OPTIMIZATION STUDY
# ==========================================
print("🔥 Starting Optuna Bayesian Search...")

# We maximize the Dice Score
study = optuna.create_study(
    direction='maximize', 
    pruner=optuna.pruners.HyperbandPruner()
)

study.optimize(
    objective, 
    n_trials=Config.N_TRIALS, 
    timeout=Config.TIMEOUT_HOURS * 3600
)

print("\n🏆 OPTIMIZATION COMPLETE")
print(f"Best Dice Score: {study.best_value:.4f}")
print("Best Hyperparameters:")
for key, value in study.best_params.items():
    print(f"   {key}: {value}")

# Save for reporting
df = study.trials_dataframe()
df.to_csv('optuna_results_hard_mining.csv', index=False)

[I 2025-12-11 16:08:45,510] A new study created in memory with name: no-name-bdfe8523-3916-4ca4-a1e6-35c39e143955


🔥 Starting Optuna Bayesian Search...


I0000 00:00:1765469326.514112      19 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


Epoch 1/15


I0000 00:00:1765469345.425834      78 service.cc:148] XLA service 0x7b29800091c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1765469345.426518      78 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1765469347.242996      78 cuda_dnn.cc:529] Loaded cuDNN version 90300


...

Epoch 1/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 53s 152ms/step - dice_metric: 0.4949 - loss: 0.4885 - val_dice_metric: 0.5338 - val_loss: 0.4399
Epoch 2/15
⚠️ Trial 28 skipped: OOM (Model too large for GPU).


[I 2025-12-11 17:57:04,511] Trial 28 pruned. 


Epoch 1/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 40s 114ms/step - dice_metric: 0.5010 - loss: 0.4510 - val_dice_metric: 0.3491 - val_loss: 0.8063
Epoch 2/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 20s 102ms/step - dice_metric: 0.6102 - loss: 0.3265 - val_dice_metric: 0.6168 - val_loss: 0.3197
Epoch 3/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 20s 102ms/step - dice_metric: 0.6327 - loss: 0.3057 - val_dice_metric: 0.6243 - val_loss: 0.3262
Epoch 4/15
200/200 ━━━━━━━━━━━━━━━━━━━━ 0s 94ms/step - dice_metric: 0.6402 - loss: 0.3004⚠️ Trial 29 failed: Trial was pruned at epoch 3.


[I 2025-12-11 17:58:56,178] Trial 29 pruned. Trial was pruned at epoch 3.



🏆 OPTIMIZATION COMPLETE
Best Dice Score: 0.6796
Best Hyperparameters:
   learning_rate: 0.00046021236610849404
   dice_weight: 0.9065980612124334
   start_filters: 64
   dropout: 0.20612200428532657
   depth: 4
